# Neuro-Symbolic Disease Prediction System V1.2

## 1. Environment Setup

In [2]:
!pip install neo4j node2vec networkx rich

In [3]:
import numpy as np
import pandas as pd
import torch
import sklearn
import neo4j

print("Package Versions:")
print(f"  NumPy: {np.__version__}")
print(f"  Pandas: {pd.__version__}")
print(f"  PyTorch: {torch.__version__}")
print(f"  Scikit-learn: {sklearn.__version__}")
print(f"  Neo4j: {neo4j.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
print("\nAll packages loaded successfully.")

Package Versions:
  NumPy: 1.26.4
  Pandas: 2.2.2
  PyTorch: 2.8.0+cu126
  Scikit-learn: 1.6.1
  Neo4j: 6.0.2
  CUDA available: True
  GPU: Tesla T4

All packages loaded successfully.


## 2. Imports & Configuration

In [4]:
import os
import json
import math
import random
import warnings
import re
import pickle
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from neo4j import GraphDatabase
from node2vec import Node2Vec
from rich import print as rprint
from rich.table import Table
from rich.console import Console
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, top_k_accuracy_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Path configuration
if 'COLAB_GPU' in os.environ or 'google.colab' in str(get_ipython()):
    DATA_DIR = Path('/content/models/neuro_symbolic_v1_2')
    WEIGHTS_PATH = Path('/content/disease_symptom_weights.csv')
    IS_COLAB = True
else:
    NOTEBOOK_DIR = Path.cwd()
    PROJECT_ROOT = NOTEBOOK_DIR.parent.parent if 'scripts/neuro-symbolic' in str(NOTEBOOK_DIR) else NOTEBOOK_DIR
    DATA_DIR = PROJECT_ROOT / 'models' / 'neuro_symbolic_v1_2'
    WEIGHTS_PATH = PROJECT_ROOT / 'scripts' / 'knowledge_base' / 'disease_database.parquet'
    if not WEIGHTS_PATH.exists():
        WEIGHTS_PATH = PROJECT_ROOT / 'scripts' / 'knowledge_base' / 'data_preparation' / 'disease_symptom_weights.csv'
    IS_COLAB = False

DATA_DIR.mkdir(parents=True, exist_ok=True)

# Neo4j configuration
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://8.tcp.ngrok.io:18944')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')

print(f"Device: {device}")
print(f"Environment: {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Data directory: {DATA_DIR}")
print(f"Weights file: {WEIGHTS_PATH}")
print(f"Weights exists: {WEIGHTS_PATH.exists()}")

Device: cuda
Environment: Google Colab
Data directory: /content/models/neuro_symbolic_v1_2
Weights file: /content/disease_symptom_weights.csv
Weights exists: True


## 3. Utility Functions

In [5]:
def make_symptom_id(symptom: str) -> str:
    """Create stable identifier for symptoms."""
    slug = re.sub(r'[^a-z0-9]+', '_', symptom.lower()).strip('_')
    if not slug:
        slug = f"symptom_{abs(hash(symptom))}"
    return f"symptom::{slug}"

## 4. Dataset Builder

In [6]:
class NeuroSymbolicDatasetBuilder:
    """Build datasets from Neo4j with automatic schema detection."""

    def __init__(self, uri: str, user: str, password: str, seed: int = 42):
        try:
            self.driver = GraphDatabase.driver(uri, auth=(user, password))
            with self.driver.session() as session:
                session.run("RETURN 1")
        except Exception as e:
            raise ConnectionError(f"Failed to connect to Neo4j: {e}")
        self.random = random.Random(seed)
        self.seed = seed

    def close(self):
        if self.driver:
            self.driver.close()

    def fetch_relations(self, min_weight: float = 0.4) -> pd.DataFrame:
        """Fetch relations with automatic schema detection."""
        # Try multiple property names
        queries = [
            # Standard query
            """
            MATCH (d:Disease)-[r:HAS_SYMPTOM]->(s:Symptom)
            WHERE r.weight >= $min_weight
            RETURN d.id AS disease_id, d.name AS disease_name,
                   s.id AS symptom_id, coalesce(s.name_display, s.name) AS symptom_name,
                   coalesce(s.name, s.id) AS symptom_key,
                   r.weight AS weight,
                   coalesce(r.tfidf_weight, r.weight) AS tfidf_weight,
                   coalesce(r.pmi_confidence, r.weight) AS pmi_confidence
            ORDER BY disease_id, weight DESC
            """,
            # Fallback with coalesce
            """
            MATCH (d:Disease)-[r:HAS_SYMPTOM]->(s:Symptom)
            WHERE r.weight >= $min_weight
            RETURN coalesce(d.disease_id, d.id, d.name) AS disease_id,
                   d.name AS disease_name,
                   coalesce(s.symptom_id, s.id) AS symptom_id,
                   coalesce(s.name_display, s.name, s.id) AS symptom_name,
                   coalesce(s.name, s.id) AS symptom_key,
                   r.weight AS weight,
                   coalesce(r.tfidf_weight, r.weight) AS tfidf_weight,
                   coalesce(r.pmi_confidence, r.weight) AS pmi_confidence
            ORDER BY disease_id, weight DESC
            """
        ]

        df = None
        for i, query in enumerate(queries):
            try:
                with self.driver.session() as session:
                    records = session.run(query, min_weight=min_weight)
                    df = pd.DataFrame(records.data())
                if not df.empty and df['disease_id'].notna().any():
                    break
            except Exception as e:
                if i == len(queries) - 1:
                    raise RuntimeError(f"All queries failed: {e}")

        if df is None or df.empty:
            raise ValueError(f"No relations found with min_weight={min_weight}")

        # Filter null disease_ids
        df = df[df['disease_id'].notna()]

        print(f"Fetched {len(df)} relations for {df['disease_id'].nunique()} diseases")
        return df

    def _sample_encounter(self, symptoms: pd.DataFrame, drop_p: float = 0.15, noise_p: float = 0.1) -> Tuple[List[str], List[str]]:
        positive = []
        for _, row in symptoms.iterrows():
            keep_prob = min(0.95, row['weight'] * 1.5)
            if self.random.random() < keep_prob:
                positive.append(row['symptom_id'])

        if len(positive) < 2:
            positive = symptoms.head(min(3, len(symptoms)))['symptom_id'].tolist()

        positive = [s for s in positive if self.random.random() > drop_p]

        if len(positive) < 2:
            positive = symptoms.sample(n=min(2, len(symptoms)), random_state=self.seed)['symptom_id'].tolist()

        noise_candidates = symptoms.sort_values('tfidf_weight', ascending=False)['symptom_id'].tolist()
        negatives = [c for c in noise_candidates if c not in positive and self.random.random() < noise_p]

        return positive, negatives

    def build(self, min_weight: float = 0.4, samples_per_disease: int = 70, drop_p: float = 0.15, noise_p: float = 0.1) -> pd.DataFrame:
        relations = self.fetch_relations(min_weight=min_weight)
        encounters = []

        for (disease_id, disease_name), group in tqdm(relations.groupby(['disease_id', 'disease_name']), desc='Generating encounters'):
            for idx in range(samples_per_disease):
                positive, negatives = self._sample_encounter(group, drop_p, noise_p)
                encounters.append({
                    'encounter_id': f"{disease_id}_{idx:04d}",
                    'disease_id': disease_id,
                    'disease_name': disease_name,
                    'positive_symptoms': positive,
                    'negative_symptoms': negatives
                })

        df = pd.DataFrame(encounters)
        print(f"Generated {len(df)} encounters for {df['disease_id'].nunique()} diseases")
        return df

    def split(self, df: pd.DataFrame, val_size: float = 0.15, test_size: float = 0.15) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        try:
            train_df, temp_df = train_test_split(df, test_size=val_size + test_size, stratify=df['disease_id'], random_state=self.seed)
            val_df, test_df = train_test_split(temp_df, test_size=test_size/(val_size+test_size), stratify=temp_df['disease_id'], random_state=self.seed)
        except:
            print("Warning: Stratified split failed, using random split")
            train_df, temp_df = train_test_split(df, test_size=val_size + test_size, random_state=self.seed)
            val_df, test_df = train_test_split(temp_df, test_size=test_size/(val_size+test_size), random_state=self.seed)

        print(f"Split: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
        return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

## 5. Load Weight Matrix

In [7]:
def load_weight_matrix(file_path: Path) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(f"Weights file not found: {file_path}")

    if file_path.suffix == '.parquet':
        weights_df = pd.read_parquet(file_path)
    else:
        weights_df = pd.read_csv(file_path)

    # Standardize columns
    if 'frequency_weight' not in weights_df.columns and 'weight' in weights_df.columns:
        weights_df['frequency_weight'] = weights_df['weight']
    if 'tfidf_weight' not in weights_df.columns:
        weights_df['tfidf_weight'] = weights_df.get('frequency_weight', 0.5)
    if 'pmi_weight' not in weights_df.columns:
        weights_df['pmi_weight'] = weights_df.get('frequency_weight', 0.5)

    weights_df['symptom_key'] = weights_df['symptom'].str.lower().str.strip()
    weights_df['symptom_id'] = weights_df['symptom_key'].apply(make_symptom_id)

    print(f"Loaded {len(weights_df)} relations ({weights_df['disease_id'].nunique()} diseases, {weights_df['symptom_id'].nunique()} symptoms)")
    return weights_df

weights_df = load_weight_matrix(WEIGHTS_PATH)

Loaded 57833 relations (9590 diseases, 16800 symptoms)


## 6. Graph Feature Extraction (Optimized)

In [8]:
class GraphFeatureExtractor:
    def __init__(self, weights: pd.DataFrame):
        self.weights = weights

    def build_graph(self) -> nx.Graph:
        graph = nx.Graph()
        edges = [(f"d::{row['disease_id']}", f"s::{row['symptom_key']}",
                 {'weight': row['frequency_weight'], 'tfidf': row['tfidf_weight'], 'pmi': row['pmi_weight']})
                for _, row in self.weights.iterrows()]
        graph.add_edges_from(edges)
        print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
        return graph

    def compute_embeddings(self, dimensions: int = 96, walk_length: int = 20, num_walks: int = 80, use_cache: bool = True):
        cache_file = DATA_DIR / f'graph_embeddings_d{dimensions}_w{num_walks}_l{walk_length}.pkl'

        if use_cache and cache_file.exists():
            print(f"Loading cached embeddings from {cache_file.name}")
            with open(cache_file, 'rb') as f:
                cached = pickle.load(f)
            print(f"Loaded {len(cached['disease'])} diseases, {len(cached['symptom'])} symptoms")
            return cached['disease'], cached['symptom']

        graph = self.build_graph()
        print(f"Computing Node2Vec embeddings (dimensions={dimensions}, walks={num_walks}, walk_length={walk_length})")

        node2vec = Node2Vec(graph, dimensions=dimensions, walk_length=walk_length, num_walks=num_walks,
                           workers=min(os.cpu_count() or 2, 4), p=1.0, q=1.0, weight_key='weight', quiet=False)
        model = node2vec.fit(window=5, min_count=1, batch_words=4, epochs=5, sg=1, workers=min(os.cpu_count() or 2, 4))

        disease_emb = {node.split('::', 1)[1]: model.wv[node] for node in graph.nodes if node.startswith('d::')}
        symptom_emb = {node.split('::', 1)[1]: model.wv[node] for node in graph.nodes if node.startswith('s::')}

        disease_df = pd.DataFrame([{'disease_id': k, 'embedding': v} for k, v in disease_emb.items()])
        symptom_df = pd.DataFrame([{'symptom_key': k, 'symptom_id': make_symptom_id(k), 'embedding': v}
                                  for k, v in symptom_emb.items()])

        if use_cache:
            with open(cache_file, 'wb') as f:
                pickle.dump({'disease': disease_df, 'symptom': symptom_df}, f)
            print(f"Cached embeddings to {cache_file.name}")

        print(f"Computed {len(disease_df)} disease embeddings, {len(symptom_df)} symptom embeddings")
        del graph, node2vec, model
        return disease_df, symptom_df

extractor = GraphFeatureExtractor(weights_df)
disease_embeds_df, symptom_embeds_df = extractor.compute_embeddings(dimensions=96, walk_length=20, num_walks=80)

Graph: 26489 nodes, 57820 edges
Computing Node2Vec embeddings (dimensions=96, walks=80, walk_length=20)


Computing transition probabilities:   0%|          | 0/26489 [00:00<?, ?it/s]

Cached embeddings to graph_embeddings_d96_w80_l20.pkl
Computed 9590 disease embeddings, 16899 symptom embeddings


## 7. Vocabularies

In [9]:
class SymptomVocabulary:
    def __init__(self):
        self.symptom_to_id: Dict[str, int] = {}
        self.id_to_symptom: List[str] = []

    def fit(self, symptom_lists: Iterable[List[str]]):
        unique = set()
        for symptoms in symptom_lists:
            unique.update(symptoms)
        self.id_to_symptom = ['<PAD>'] + sorted(unique)
        self.symptom_to_id = {s: i for i, s in enumerate(self.id_to_symptom)}

    def transform(self, symptoms: List[str]) -> List[int]:
        return [self.symptom_to_id.get(s, 0) for s in symptoms]

    def __len__(self):
        return len(self.id_to_symptom)

class DiseaseVocabulary:
    def __init__(self):
        self.label_encoder = LabelEncoder()

    def fit(self, diseases: Iterable[str]):
        self.label_encoder.fit(list(diseases))

    def transform(self, diseases: Iterable[str]) -> np.ndarray:
        return self.label_encoder.transform(list(diseases))

    def inverse_transform(self, labels: Iterable[int]) -> List[str]:
        return list(self.label_encoder.inverse_transform(list(labels)))

    def __len__(self):
        return len(self.label_encoder.classes_)

## 8. Dataset & DataLoader

In [10]:
class EncounterDataset(Dataset):
    def __init__(self, df: pd.DataFrame, symptom_vocab: SymptomVocabulary, disease_vocab: DiseaseVocabulary):
        self.df = df
        self.symptom_vocab = symptom_vocab
        self.disease_vocab = disease_vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        all_symptoms = row['positive_symptoms'] + row['negative_symptoms']
        return {
            'symptom_ids': self.symptom_vocab.transform(all_symptoms),
            'positives': row['positive_symptoms'],
            'label': self.disease_vocab.transform([row['disease_id']])[0],
            'vocab_size': len(self.symptom_vocab)
        }

def create_collate_fn(symptom_vocab: SymptomVocabulary):
    def collate_batch(batch):
        max_len = max(len(item['symptom_ids']) for item in batch)
        vocab_size = batch[0]['vocab_size']

        padded = [item['symptom_ids'] + [0] * (max_len - len(item['symptom_ids'])) for item in batch]

        bags = []
        for item in batch:
            bag = torch.zeros(vocab_size, dtype=torch.float32)
            for symptom in item['positives']:
                idx = symptom_vocab.symptom_to_id.get(symptom, 0)
                if idx:
                    bag[idx] = 1.0
            bags.append(bag)

        return {
            'symptom_ids': torch.tensor(padded, dtype=torch.long),
            'bag_vector': torch.stack(bags),
            'labels': torch.tensor([item['label'] for item in batch], dtype=torch.long)
        }
    return collate_batch

## 9. Neural Architecture (Enhanced)

In [11]:
class KnowledgeGraphPrior(nn.Module):
    def __init__(self, weight_matrix: torch.Tensor):
        super().__init__()
        self.register_buffer('weight_matrix', weight_matrix)

    def forward(self, bag_vector: torch.Tensor) -> torch.Tensor:
        return torch.matmul(bag_vector, self.weight_matrix)

class SymptomSetEncoder(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, symptom_embedding_matrix: np.ndarray, dropout: float = 0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        with torch.no_grad():
            init_tensor = torch.tensor(symptom_embedding_matrix, dtype=torch.float32)
            if init_tensor.shape[0] != vocab_size or init_tensor.shape[1] != embed_dim:
                init_tensor = torch.randn(vocab_size, embed_dim) * 0.01
                for i, emb in enumerate(symptom_embedding_matrix[:min(vocab_size, len(symptom_embedding_matrix))]):
                    init_tensor[i, :len(emb)] = torch.tensor(emb[:embed_dim])
            self.embedding.weight.copy_(init_tensor)

        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, dim_feedforward=embed_dim*2,
                                                   dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.cls = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, symptom_ids: torch.Tensor) -> torch.Tensor:
        embeddings = self.embedding(symptom_ids)
        mask = symptom_ids == 0
        encoded = self.encoder(embeddings, src_key_padding_mask=mask)
        pooled = encoded[:, 0, :] if encoded.size(1) > 0 else encoded.mean(dim=1)
        return self.dropout(torch.tanh(self.cls(pooled)))

class NeuralPredictor(nn.Module):
    def __init__(self, input_dim: int, num_diseases: int, graph_disease_matrix: np.ndarray, hidden_dim: int = 256, dropout: float = 0.3):
        super().__init__()
        bias = torch.zeros(num_diseases)
        if graph_disease_matrix.shape[0] == num_diseases:
            bias = torch.tensor(graph_disease_matrix.mean(axis=1), dtype=torch.float32)
        self.pre_bias = nn.Parameter(bias)

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_diseases)
        )

    def forward(self, encoded: torch.Tensor) -> torch.Tensor:
        return self.mlp(encoded) + self.pre_bias

class FusionGate(nn.Module):
    def __init__(self, num_diseases: int):
        super().__init__()
        self.controller = nn.Sequential(
            nn.Linear(num_diseases * 3, num_diseases), nn.ReLU(),
            nn.Linear(num_diseases, 3)
        )

    def forward(self, neural_logits, symbolic_logits, walk_logits):
        stacked = torch.stack([neural_logits, symbolic_logits, walk_logits], dim=1)
        weights = torch.softmax(self.controller(stacked.reshape(stacked.size(0), -1)), dim=1)
        return (weights.unsqueeze(-1) * stacked).sum(dim=1), weights

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1))

    def forward(self, logits):
        return logits / self.temperature.clamp(0.5, 5.0)

class HybridNeuroSymbolicModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_diseases, symptom_embedding_matrix,
                 disease_prior_matrix, kg_weight_matrix, dropout=0.3):
        super().__init__()
        self.symptom_encoder = SymptomSetEncoder(vocab_size, embed_dim, symptom_embedding_matrix, dropout)
        self.neural_predictor = NeuralPredictor(embed_dim, num_diseases, disease_prior_matrix, 256, dropout)
        self.kg_prior = KnowledgeGraphPrior(kg_weight_matrix)
        self.walk_prior_projection = nn.Linear(kg_weight_matrix.shape[1], num_diseases, bias=False)
        self.fusion_gate = FusionGate(num_diseases)
        self.temperature = TemperatureScaler()

    def forward(self, symptom_ids, bag_vector):
        encoded = self.symptom_encoder(symptom_ids)
        neural_logits = self.neural_predictor(encoded)
        symbolic_logits = self.kg_prior(bag_vector)
        walk_logits = self.walk_prior_projection(symbolic_logits)
        fused_logits, weights = self.fusion_gate(neural_logits, symbolic_logits, walk_logits)
        calibrated_logits = self.temperature(fused_logits)
        return {'calibrated_logits': calibrated_logits, 'fusion_weights': weights}

    def predict_proba(self, symptom_ids, bag_vector):
        return torch.softmax(self.forward(symptom_ids, bag_vector)['calibrated_logits'], dim=-1)

## 10. Training Functions

In [ ]:
def compute_metrics(logits, labels, topk=(1, 3, 5)):
    preds = torch.argmax(logits, dim=1)
    metrics = {
        'accuracy': accuracy_score(labels.cpu(), preds.cpu()),
        'f1_macro': f1_score(labels.cpu(), preds.cpu(), average='macro', zero_division=0)
    }
    probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
    for k in topk:
        try:
            metrics[f'top{k}_accuracy'] = top_k_accuracy_score(labels.cpu(), probs, k=k, labels=np.arange(probs.shape[1]))
        except:
            metrics[f'top{k}_accuracy'] = float('nan')
    return metrics

def train_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        symptom_ids = batch['symptom_ids'].to(device)
        bag_vector = batch['bag_vector'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(symptom_ids, bag_vector)
        loss = criterion(outputs['calibrated_logits'], labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item() * symptom_ids.size(0)
    return total_loss / len(dataloader.dataset)

def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            symptom_ids = batch['symptom_ids'].to(device)
            bag_vector = batch['bag_vector'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(symptom_ids, bag_vector)
            loss = criterion(outputs['calibrated_logits'], labels)

            total_loss += loss.item() * symptom_ids.size(0)
            all_logits.append(outputs['calibrated_logits'].cpu())
            all_labels.append(labels.cpu())

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    metrics = compute_metrics(logits, labels)
    return total_loss / len(dataloader.dataset), metrics

def train_model(model, train_loader, val_loader, epochs=100, lr=1e-3, weight_decay=1e-4, patience=10, warmup_epochs=3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.0)

    def warmup_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        return 1.0

    warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_lambda)
    reduce_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_acc': []}
    best_val_loss, best_state, patience_counter = float('inf'), None, 0

    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_metrics = evaluate(model, val_loader, criterion)

        if epoch <= warmup_epochs:
            warmup_scheduler.step()
        else:
            reduce_scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_metrics['f1_macro'])
        history['val_acc'].append(val_metrics['accuracy'])

        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:02d} | LR: {current_lr:.6f} | Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_metrics['accuracy']:.4f} | Val F1: {val_metrics['f1_macro']:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch}')
                break

    if best_state:
        model.load_state_dict(best_state)
    return history

## 11. Complete Training Pipeline

In [13]:
def run_training_pipeline():
    print("NEURO-SYMBOLIC MODEL TRAINING V1.2\n")

    # Build dataset from Neo4j
    print("[1/7] Building dataset from Neo4j")
    builder = NeuroSymbolicDatasetBuilder(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, SEED)
    try:
        encounters_df = builder.build(min_weight=0.5, samples_per_disease=70)
        train_df, val_df, test_df = builder.split(encounters_df)
    finally:
        builder.close()

    # Build vocabularies
    print("\n[2/7] Building vocabularies")
    symptom_vocab = SymptomVocabulary()
    symptom_vocab.fit(train_df['positive_symptoms'])
    disease_vocab = DiseaseVocabulary()
    disease_vocab.fit(train_df['disease_id'])
    print(f"Vocabulary sizes - Symptoms: {len(symptom_vocab)}, Diseases: {len(disease_vocab)}")

    # Align embeddings to vocabularies
    print("\n[3/7] Aligning embeddings to vocabularies")
    symptom_emb_matrix = np.random.randn(len(symptom_vocab), 96) * 0.01
    symptom_lookup = {row['symptom_id']: row['embedding'] for _, row in symptom_embeds_df.iterrows()}
    for symptom_id, idx in symptom_vocab.symptom_to_id.items():
        if symptom_id in symptom_lookup:
            symptom_emb_matrix[idx] = symptom_lookup[symptom_id][:96]

    disease_lookup = {row['disease_id']: row['embedding'] for _, row in disease_embeds_df.iterrows()}
    disease_prior_matrix = np.stack([disease_lookup.get(label, np.zeros(96))
                                     for label in disease_vocab.label_encoder.classes_])

    # Build knowledge graph prior matrix
    print("\n[4/7] Building knowledge graph prior matrix")
    kg_matrix = torch.zeros(len(symptom_vocab), len(disease_vocab))
    for idx, disease_id in enumerate(disease_vocab.label_encoder.classes_):
        relations = weights_df[weights_df['disease_id'] == disease_id]
        for _, row in relations.iterrows():
            symptom_idx = symptom_vocab.symptom_to_id.get(row['symptom_id'], 0)
            if symptom_idx > 0:
                kg_matrix[symptom_idx, idx] = row['tfidf_weight']

    # Create PyTorch datasets and dataloaders
    print("\n[5/7] Creating datasets and dataloaders")
    train_dataset = EncounterDataset(train_df, symptom_vocab, disease_vocab)
    val_dataset = EncounterDataset(val_df, symptom_vocab, disease_vocab)
    test_dataset = EncounterDataset(test_df, symptom_vocab, disease_vocab)

    collate_fn = create_collate_fn(symptom_vocab)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

    # Initialize model
    print("\n[6/7] Initializing hybrid neuro-symbolic model")
    model = HybridNeuroSymbolicModel(
        vocab_size=len(symptom_vocab),
        embed_dim=96,
        num_diseases=len(disease_vocab),
        symptom_embedding_matrix=symptom_emb_matrix,
        disease_prior_matrix=disease_prior_matrix,
        kg_weight_matrix=kg_matrix
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")

    # Train model with improved hyperparameters
    print("\n[7/7] Training model")
    print("Training configuration:")
    print("  Learning rate: 1e-3 (with 3-epoch warmup)")
    print("  Max epochs: 100")
    print("  Early stopping patience: 10")
    print("  Label smoothing: 0.0 (disabled)")
    print("  Gradient clipping: 5.0")
    print("  LR scheduling: Warmup + ReduceLROnPlateau\n")

    history = train_model(model, train_loader, val_loader, epochs=100, lr=1e-3, patience=10, warmup_epochs=3)

    # Save model and vocabularies
    torch.save(model.state_dict(), DATA_DIR / 'model.pt')
    torch.save({'symptom_vocab': symptom_vocab, 'disease_vocab': disease_vocab,
                'model_config': {'vocab_size': len(symptom_vocab), 'embed_dim': 96, 'num_diseases': len(disease_vocab)}},
               DATA_DIR / 'vocabularies.pt')
    print(f"\nModel saved to: {DATA_DIR}")

    # Evaluate on test set
    print("\nEvaluating on test set")
    _, test_metrics = evaluate(model, test_loader, nn.CrossEntropyLoss())
    for k, v in test_metrics.items():
        print(f"  {k}: {v:.4f}")

    print(f"\nTraining complete. Model files saved to {DATA_DIR}")

    return model, symptom_vocab, disease_vocab, history

model, symptom_vocab, disease_vocab, history = run_training_pipeline()

NEURO-SYMBOLIC MODEL TRAINING V1.2

[1/7] Building dataset from Neo4j
Fetched 3121 relations for 1918 diseases


Generating encounters:   0%|          | 0/1918 [00:00<?, ?it/s]

Generated 134260 encounters for 1918 diseases
Split: train=93982, val=20139, test=20139

[2/7] Building vocabularies
Vocabulary sizes - Symptoms: 2908, Diseases: 1918

[3/7] Aligning embeddings to vocabularies

[4/7] Building knowledge graph prior matrix

[5/7] Creating datasets and dataloaders

[6/7] Initializing hybrid neuro-symbolic model
Total parameters: 15,467,688

[7/7] Training model
Training configuration:
  Learning rate: 1e-3 (with 3-epoch warmup)
  Max epochs: 100
  Early stopping patience: 10
  Label smoothing: 0.0 (disabled)
  Gradient clipping: 5.0
  LR scheduling: Warmup + ReduceLROnPlateau



TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

## 12. Prediction Functions

In [ ]:
def predict_disease(model, symptoms: List[str], symptom_vocab, disease_vocab, top_k=5):
    """Predict diseases from symptoms with OOV handling."""
    model.eval()
    
    # Convert symptoms to IDs
    symptom_ids = [make_symptom_id(s) for s in symptoms]
    
    # Check which symptoms are in vocabulary
    valid_symptoms = [sid for sid in symptom_ids if sid in symptom_vocab.symptom_to_id and symptom_vocab.symptom_to_id[sid] > 0]
    
    if len(valid_symptoms) == 0:
        print(f"WARNING: None of the provided symptoms are in the vocabulary!")
        print(f"Provided: {symptoms}")
        print(f"Try using symptom names from the training data.")
        # Return uniform distribution as fallback
        num_diseases = len(disease_vocab)
        uniform_prob = 1.0 / num_diseases
        top_indices = list(range(min(top_k, num_diseases)))
        return {
            'predictions': [(disease_vocab.inverse_transform([idx])[0], uniform_prob) for idx in top_indices],
            'warning': 'All symptoms out-of-vocabulary'
        }
    
    # Build input tensors with valid symptoms only
    encoded_ids = torch.tensor([symptom_vocab.transform(valid_symptoms)], dtype=torch.long).to(device)
    
    # Build bag-of-symptoms vector
    bag = torch.zeros((1, len(symptom_vocab)), dtype=torch.float32).to(device)
    for sid in valid_symptoms:
        idx = symptom_vocab.symptom_to_id.get(sid, 0)
        if idx > 0:
            bag[0, idx] = 1.0
    
    # Get predictions
    with torch.no_grad():
        probs = model.predict_proba(encoded_ids, bag).cpu().numpy()[0]
    
    top_indices = probs.argsort()[-top_k:][::-1]
    return {
        'predictions': [(disease_vocab.inverse_transform([idx])[0], float(probs[idx])) for idx in top_indices],
        'matched_symptoms': len(valid_symptoms),
        'total_symptoms': len(symptoms)
    }

def load_trained_model(model_dir=DATA_DIR):
    """Load saved model."""
    vocab_data = torch.load(model_dir / 'vocabularies.pt', map_location=device, weights_only=False)
    symptom_vocab = vocab_data['symptom_vocab']
    disease_vocab = vocab_data['disease_vocab']
    config = vocab_data['model_config']

    model = HybridNeuroSymbolicModel(
        vocab_size=config['vocab_size'],
        embed_dim=config['embed_dim'],
        num_diseases=config['num_diseases'],
        symptom_embedding_matrix=np.random.randn(config['vocab_size'], config['embed_dim']) * 0.01,
        disease_prior_matrix=np.zeros((config['num_diseases'], config['embed_dim'])),
        kg_weight_matrix=torch.zeros(config['vocab_size'], config['num_diseases'])
    ).to(device)

    model.load_state_dict(torch.load(model_dir / 'model.pt', map_location=device, weights_only=True))
    model.eval()
    return model, symptom_vocab, disease_vocab

def show_available_symptoms(symptom_vocab, n=20):
    """Display sample symptoms from vocabulary for reference."""
    symptoms = [s for s in symptom_vocab.id_to_symptom[1:n+1]]  # Skip <PAD>
    print(f"\nSample symptoms in vocabulary (showing {min(n, len(symptoms))} of {len(symptom_vocab)-1}):")
    for i, s in enumerate(symptoms, 1):
        # Remove symptom:: prefix for display
        display_name = s.replace('symptom::', '').replace('_', ' ')
        print(f"  {i}. {display_name}")
    print(f"\nUse these format when making predictions.")

## 13. Training Performance Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', marker='o', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', marker='s', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss', fontsize=11)
axes[0].set_title('Training & Validation Loss', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['val_acc'], label='Val Accuracy', color='blue', marker='o', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Accuracy', fontsize=11)
axes[1].set_title('Validation Accuracy', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1 plot
axes[2].plot(history['val_f1'], label='Val F1', color='green', marker='o', linewidth=2)
axes[2].set_xlabel('Epoch', fontsize=11)
axes[2].set_ylabel('F1 Score', fontsize=11)
axes[2].set_title('Validation F1 Score', fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTraining Summary:")
print(f"  Best Val Accuracy: {max(history['val_acc']):.4f}")
print(f"  Best Val F1: {max(history['val_f1']):.4f}")
print(f"  Best Val Loss: {min(history['val_loss']):.4f}")
print(f"  Final Train Loss: {history['train_loss'][-1]:.4f}")
print(f"  Final Val Loss: {history['val_loss'][-1]:.4f}")

## 14. Prediction Examples

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/models/neuro_symbolic_v1_2 /content/drive/MyDrive/MSc/NeuroSymbolicReasonerV1.2

In [ ]:
print("Loading saved model...")
loaded_model, loaded_symptom_vocab, loaded_disease_vocab = load_trained_model(DATA_DIR)
print("Model loaded successfully.\n")

# Show available symptoms in vocabulary
show_available_symptoms(loaded_symptom_vocab, n=30)

# Get actual symptoms from vocabulary for realistic predictions
available_symptoms = [s.replace('symptom::', '').replace('_', ' ') 
                     for s in loaded_symptom_vocab.id_to_symptom[1:100]]

# Example predictions with symptoms from the vocabulary
# These should match symptoms from your Neo4j database
test_cases = [
    # Use symptoms that are likely in your knowledge base
    # Format: exact match to how they appear in vocabulary
    [available_symptoms[0], available_symptoms[1], available_symptoms[2]],
    [available_symptoms[5], available_symptoms[6]],
    [available_symptoms[10], available_symptoms[11], available_symptoms[12], available_symptoms[13]],
]

print("\n" + "="*60)
print("DISEASE PREDICTIONS")
print("="*60)

for i, symptoms in enumerate(test_cases, 1):
    print(f"\nCase {i}:")
    results = predict_disease(loaded_model, symptoms, loaded_symptom_vocab, loaded_disease_vocab, top_k=5)
    
    if 'warning' in results:
        print(f"  ⚠️ {results['warning']}")
    else:
        print(f"  Symptoms ({results['matched_symptoms']}/{results['total_symptoms']} matched): {', '.join(symptoms)}")
    
    print(f"  Top predictions:")
    for j, (disease, conf) in enumerate(results['predictions'], 1):
        print(f"    {j}. {disease}: {conf:.2%}")

print("\n" + "="*60)
print("\nTo make custom predictions, use symptom names from the vocabulary above.")